In [ ]:
# Blok Kode untuk Menghasilkan 1000 Data Kotor
# Cukup jalankan sel ini satu kali.

import pandas as pd
import numpy as np
import random
import io

# Coba impor Faker, jika tidak ada, instal
try:
    from faker import Faker
except ImportError:
    print("Faker not found. Installing...")
    !pip install Faker
    from faker import Faker

# Inisialisasi Faker untuk data Indonesia
fake = Faker('id_ID')

# --- Parameter untuk generasi data ---
num_rows = 1000
pendidikan_options = ['SMA', 'SMK', 'D3', 'S1', 'S2', 'Sarjana', 'Bachelor', 'Diploma', 'S1', 'S1'] # S1 sengaja dibuat lebih banyak
kota_options = ['Jakarta', 'Bandung', 'Surabaya', 'Yogyakarta', 'Medan', 'Semarang', 'Makassar', 'Palembang', 'Bandar Lampung', 'Denpasar']
base_salary = {'SMA': 4500000, 'SMK': 4800000, 'D3': 6000000, 'Diploma': 6200000, 'S1': 8000000, 'Sarjana': 8000000, 'Bachelor': 8000000, 'S2': 12000000}

# --- Proses Generasi Data ---
data_list = []
for i in range(num_rows):
    pendidikan = random.choice(pendidikan_options)
    umur = random.randint(21, 58)
    lama_kerja = max(1, umur - random.randint(19, 23))
    gaji = base_salary[pendidikan] + (lama_kerja * random.randint(250000, 600000))

    record = {
        'ID': i + 1,
        'Nama': fake.name(),
        'Umur': umur,
        'Pendidikan': pendidikan,
        'Kota': random.choice(kota_options),
        'Lama_Kerja': lama_kerja,
        'Gaji': int(gaji / 1000) * 1000 # Pembulatan gaji
    }
    data_list.append(record)

df = pd.DataFrame(data_list)

# --- Proses "Mengotori" Data ---

# 1. Menambahkan Data Hilang (NaN)
num_missing = int(num_rows * 0.1) # 10% data akan punya nilai hilang
for _ in range(num_missing):
    row_idx = random.randint(0, num_rows - 1)
    col_to_null = random.choice(['Gaji', 'Lama_Kerja', 'Kota'])
    df.loc[row_idx, col_to_null] = np.nan

# 2. Menambahkan Outliers
num_outliers = 15
for _ in range(num_outliers):
    row_idx = random.randint(0, num_rows - 1)
    # Outlier Umur
    if random.random() > 0.5:
        df.loc[row_idx, 'Umur'] = random.choice([1, 2, 150, 175, 200])
    # Outlier Gaji
    else:
        df.loc[row_idx, 'Gaji'] = random.choice([500000, 1000000, 500000000, 900000000])

# 3. Menambahkan Duplikat
num_duplicates = 40
duplicate_rows = df.sample(n=num_duplicates)
df = pd.concat([df, duplicate_rows], ignore_index=True)

# 4. Acak semua baris agar "kotoran" tersebar
df = df.sample(frac=1).reset_index(drop=True)
# Perbarui ID setelah diacak
df['ID'] = df.index + 1


# --- Konversi ke format CSV String ---
output = io.StringIO()
df.to_csv(output, index=False)
csv_data = output.getvalue()
output.close()

print(f"✅ Berhasil! Variabel 'csv_data' telah dibuat dengan {len(df)} baris data.")
print("Anda bisa melanjutkan ke sel berikutnya untuk memuat dan membersihkan data.")

# Tampilkan 10 baris pertama sebagai contoh
# df_preview = pd.read_csv(io.StringIO(csv_data))
# print("\nContoh 10 baris pertama dari data yang dihasilkan:")
# print(df_preview.head(10))

Faker not found. Installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 17.6 MB/s eta 0:00:00
✅ Berhasil! Variabel 'csv_data' telah dibuat dengan 1040 baris data.
Anda bisa melanjutkan ke sel berikutnya untuk memuat dan membersihkan data.


In [ ]:
# Simpan csv_data ke file
file_path = 'data_kotor.csv'
with open(file_path, 'w') as f:
    f.write(csv_data)

print(f"✅ Variabel 'csv_data' telah disimpan ke file '{file_path}'.")

✅ Variabel 'csv_data' telah disimpan ke file 'data_kotor.csv'.


In [ ]:
# Baca file CSV ke DataFrame
df_kotor = pd.read_csv('data_kotor.csv')

# Tampilkan informasi dasar tentang DataFrame
print("Informasi DataFrame:")
df_kotor.info()

# Tampilkan beberapa baris pertama
print("\nBeberapa baris pertama dari DataFrame:")
display(df_kotor.head())

Informasi DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          1040 non-null   int64  
 1   Nama        1040 non-null   object 
 2   Umur        1040 non-null   int64  
 3   Pendidikan  1040 non-null   object 
 4   Kota        1008 non-null   object 
 5   Lama_Kerja  1009 non-null   float64
 6   Gaji        1001 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 57.0+ KB

Beberapa baris pertama dari DataFrame:


,ID,Nama,Umur,Pendidikan,Kota,Lama_Kerja,Gaji
0,1,"drg. Puji Simanjuntak, S.Pd",28,Diploma,Makassar,5.0,8917000.0
1,2,"Drajat Usada, S.T.",23,Diploma,Denpasar,1.0,6686000.0
2,3,Devi Lazuardi,56,S1,Jakarta,37.0,28173000.0
3,4,T. Asmadi Simbolon,47,S1,Bandar Lampung,27.0,NaN
4,5,"T. Martaka Kuswoyo, M.Pd",22,Bachelor,Jakarta,1.0,8351000.0


In [ ]:
# 1. Periksa nilai yang hilang (Missing Values)
print("Jumlah nilai yang hilang per kolom:")
print(df_kotor.isnull().sum())

# 2. Periksa data duplikat
print("\nJumlah data duplikat:")
print(df_kotor.duplicated().sum())

# 3. Statistik Deskriptif untuk kolom numerik
print("\nStatistik deskriptif untuk kolom numerik:")
display(df_kotor.describe())

# 4. Statistik Deskriptif untuk kolom kategorikal (object)
print("\nStatistik deskriptif untuk kolom kategorikal:")
display(df_kotor.describe(include='object'))

# 5. Distribusi nilai unik pada kolom kategorikal (opsional, jika ingin melihat lebih detail)
print("\nDistribusi nilai unik pada kolom 'Pendidikan':")
print(df_kotor['Pendidikan'].value_counts())
print("\nDistribusi nilai unik pada kolom 'Kota':")
print(df_kotor['Kota'].value_counts())

Jumlah nilai yang hilang per kolom:
ID             0
Nama           0
Umur           0
Pendidikan     0
Kota          32
Lama_Kerja    31
Gaji          39
dtype: int64

Jumlah data duplikat:
0

Statistik deskriptif untuk kolom numerik:


,ID,Umur,Lama_Kerja,Gaji
count,1040.000000,1040.000000,1009.000000,1.001000e+03
mean,520.500000,39.468269,17.984143,1.630603e+07
std,300.366443,15.147620,11.073875,3.234759e+07
min,1.000000,1.000000,1.000000,1.000000e+06
25%,260.750000,29.000000,8.000000,1.049800e+07
50%,520.500000,38.000000,18.000000,1.460800e+07
75%,780.250000,48.000000,27.000000,1.871600e+07
max,1040.000000,200.000000,39.000000,9.000000e+08



Statistik deskriptif untuk kolom kategorikal:


,Nama,Pendidikan,Kota
count,1040,1040,1008
unique,997,8,10
top,R.A. Belinda Adriansyah,S1,Makassar
freq,2,344,114



Distribusi nilai unik pada kolom 'Pendidikan':
Pendidikan
S1          344
D3          114
Sarjana     109
S2          101
Diploma      99
SMA          96
SMK          91
Bachelor     86
Name: count, dtype: int64

Distribusi nilai unik pada kolom 'Kota':
Kota
Makassar          114
Bandar Lampung    113
Denpasar          112
Semarang          109
Jakarta           107
Bandung            97
Medan              96
Surabaya           90
Palembang          86
Yogyakarta         84
Name: count, dtype: int64


In [ ]:
# Membuat salinan DataFrame untuk operasi pembersihan
df_bersih = df_kotor.copy()

# 1. Penanganan Nilai yang Hilang (Missing Values)
# Mengisi nilai yang hilang pada kolom numerik ('Lama_Kerja', 'Gaji') dengan median
df_bersih['Lama_Kerja'] = df_bersih['Lama_Kerja'].fillna(df_bersih['Lama_Kerja'].median())
df_bersih['Gaji'] = df_bersih['Gaji'].fillna(df_bersih['Gaji'].median())

# Mengisi nilai yang hilang pada kolom kategorikal ('Kota') dengan modus (nilai yang paling sering muncul)
df_bersih['Kota'] = df_bersih['Kota'].fillna(df_bersih['Kota'].mode()[0])

print("✅ Penanganan nilai yang hilang selesai.")
print("Jumlah nilai yang hilang setelah diisi:")
print(df_bersih.isnull().sum())

# 2. Penanganan Outlier
# Outlier pada Umur: Mengganti nilai yang tidak masuk akal (misal: < 18 atau > 100) dengan median
median_umur = df_bersih['Umur'].median()
df_bersih['Umur'] = df_bersih['Umur'].apply(lambda x: median_umur if x < 18 or x > 100 else x)

# Outlier pada Gaji: Menggunakan metode IQR atau Z-score, atau secara sederhana mengganti nilai yang sangat ekstrim
# Kita akan menggunakan metode sederhana dengan mengganti nilai yang sangat besar atau sangat kecil dengan median gaji S1/Sarjana
median_gaji_s1 = df_bersih[df_bersih['Pendidikan'].isin(['S1', 'Sarjana', 'Bachelor'])]['Gaji'].median()
# Anggap gaji di bawah 3jt atau di atas 200jt sebagai outlier ekstrim
df_bersih['Gaji'] = df_bersih['Gaji'].apply(lambda x: median_gaji_s1 if x < 3000000 or x > 200000000 else x)


print("\n✅ Penanganan outlier pada Umur dan Gaji selesai.")
print("Statistik deskriptif Umur setelah penanganan outlier:")
display(df_bersih['Umur'].describe())
print("\nStatistik deskriptif Gaji setelah penanganan outlier:")
display(df_bersih['Gaji'].describe())


# 3. Penyeragaman Nilai Kategorikal
# Menyatukan kategori pendidikan yang serupa: 'Sarjana', 'Bachelor', 'S1' menjadi 'S1'
df_bersih['Pendidikan'] = df_bersih['Pendidikan'].replace(['Sarjana', 'Bachelor'], 'S1')

print("\n✅ Penyeragaman nilai pada kolom Pendidikan selesai.")
print("Distribusi nilai unik pada kolom 'Pendidikan' setelah penyeragaman:")
print(df_bersih['Pendidikan'].value_counts())

# 4. Menghapus Duplikat
# Berdasarkan eksplorasi data awal, tidak ada duplikat, tetapi kode ini tetap disiapkan
df_bersih = df_bersih.drop_duplicates()

print(f"\n✅ Penanganan duplikat selesai. Jumlah baris setelah menghapus duplikat: {len(df_bersih)}")


print("\n✅ Proses Pembersihan Data Selesai.")
print("\nInformasi DataFrame setelah Pembersihan:")
df_bersih.info()
print("\nBeberapa baris pertama dari DataFrame setelah Pembersihan:")
display(df_bersih.head())

✅ Penanganan nilai yang hilang selesai.
Jumlah nilai yang hilang setelah diisi:
ID            0
Nama          0
Umur          0
Pendidikan    0
Kota          0
Lama_Kerja    0
Gaji          0
dtype: int64

✅ Penanganan outlier pada Umur dan Gaji selesai.
Statistik deskriptif Umur setelah penanganan outlier:


,Umur
count,1040.000000
mean,38.843269
std,10.995206
min,21.000000
25%,29.000000
50%,38.000000
75%,48.000000
max,58.000000



Statistik deskriptif Gaji setelah penanganan outlier:


,Gaji
count,1.040000e+03
mean,1.496615e+07
std,5.286849e+06
min,4.809000e+06
25%,1.077300e+07
50%,1.460800e+07
75%,1.854800e+07
max,3.138100e+07



✅ Penyeragaman nilai pada kolom Pendidikan selesai.
Distribusi nilai unik pada kolom 'Pendidikan' setelah penyeragaman:
Pendidikan
S1         539
D3         114
S2         101
Diploma     99
SMA         96
SMK         91
Name: count, dtype: int64

✅ Penanganan duplikat selesai. Jumlah baris setelah menghapus duplikat: 1040

✅ Proses Pembersihan Data Selesai.

Informasi DataFrame setelah Pembersihan:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          1040 non-null   int64  
 1   Nama        1040 non-null   object 
 2   Umur        1040 non-null   float64
 3   Pendidikan  1040 non-null   object 
 4   Kota        1040 non-null   object 
 5   Lama_Kerja  1040 non-null   float64
 6   Gaji        1040 non-null   float64
dtypes: float64(3), int64(1), object(3)
memory usage: 57.0+ KB

Beberapa baris pertama dari DataFrame setelah Pembersihan

,ID,Nama,Umur,Pendidikan,Kota,Lama_Kerja,Gaji
0,1,"drg. Puji Simanjuntak, S.Pd",28.0,Diploma,Makassar,5.0,8917000.0
1,2,"Drajat Usada, S.T.",23.0,Diploma,Denpasar,1.0,6686000.0
2,3,Devi Lazuardi,56.0,S1,Jakarta,37.0,28173000.0
3,4,T. Asmadi Simbolon,47.0,S1,Bandar Lampung,27.0,14608000.0
4,5,"T. Martaka Kuswoyo, M.Pd",22.0,S1,Jakarta,1.0,8351000.0


In [ ]:
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import pandas as pd # Import pandas

# Pisahkan fitur
features = ['Umur', 'Pendidikan', 'Kota', 'Lama_Kerja', 'Gaji']
df_features = df_bersih[features].copy()

# Identifikasi kolom numerik dan kategorikal
numeric_features = ['Umur', 'Lama_Kerja', 'Gaji']
categorical_features = ['Pendidikan', 'Kota']

# --- Pra-pemrosesan Kolom Numerik (Scaling) ---
scaler = MinMaxScaler()
df_numeric_scaled = pd.DataFrame(scaler.fit_transform(df_features[numeric_features]), columns=numeric_features)

# --- Pra-pemrosesan Kolom Kategorikal (Encoding) ---
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False) # Gunakan sparse_output=False untuk mendapatkan array numpy padat
df_categorical_encoded_array = encoder.fit_transform(df_features[categorical_features])

# Dapatkan nama kolom baru setelah encoding
ohe_categories = encoder.get_feature_names_out(categorical_features)
df_categorical_encoded = pd.DataFrame(df_categorical_encoded_array, columns=ohe_categories)


# --- Menggabungkan Hasil Pra-pemrosesan ---
# Gabungkan DataFrame hasil scaling dan encoding
df_processed = pd.concat([df_numeric_scaled, df_categorical_encoded], axis=1)


print("✅ Proses Pra-pemrosesan Data Selesai.")
print("\nInformasi DataFrame setelah Pra-pemrosesan:")
df_processed.info()
print("\nBeberapa baris pertama dari DataFrame setelah Pra-pemrosesan:")
display(df_processed.head())

✅ Proses Pra-pemrosesan Data Selesai.

Informasi DataFrame setelah Pra-pemrosesan:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1040 entries, 0 to 1039
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Umur                 1040 non-null   float64
 1   Lama_Kerja           1040 non-null   float64
 2   Gaji                 1040 non-null   float64
 3   Pendidikan_D3        1040 non-null   float64
 4   Pendidikan_Diploma   1040 non-null   float64
 5   Pendidikan_S1        1040 non-null   float64
 6   Pendidikan_S2        1040 non-null   float64
 7   Pendidikan_SMA       1040 non-null   float64
 8   Pendidikan_SMK       1040 non-null   float64
 9   Kota_Bandar Lampung  1040 non-null   float64
 10  Kota_Bandung         1040 non-null   float64
 11  Kota_Denpasar        1040 non-null   float64
 12  Kota_Jakarta         1040 non-null   float64
 13  Kota_Makassar        1040 non-null   float64
 14  Kota_

,Umur,Lama_Kerja,Gaji,Pendidikan_D3,Pendidikan_Diploma,Pendidikan_S1,Pendidikan_S2,Pendidikan_SMA,Pendidikan_SMK,Kota_Bandar Lampung,Kota_Bandung,Kota_Denpasar,Kota_Jakarta,Kota_Makassar,Kota_Medan,Kota_Palembang,Kota_Semarang,Kota_Surabaya,Kota_Yogyakarta
0,0.189189,0.105263,0.154599,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0.054054,0.000000,0.070638,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.945946,0.947368,0.879271,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.702703,0.684211,0.368772,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.027027,0.000000,0.133298,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Simpan DataFrame df_bersih ke file CSV
df_bersih.to_csv('data_bersih.csv', index=False)

print("✅ DataFrame 'df_bersih' telah disimpan ke file 'data_bersih.csv'.")

✅ DataFrame 'df_bersih' telah disimpan ke file 'data_bersih.csv'.
